In [1]:
import pandas as pd
import yaml
import os
import sys
sys.path.append('/workspace/src/')
from utils.helper_utils import split_annotations

In [2]:
with open("/workspace/experiment_config.yaml", 'r') as file:
    config = yaml.safe_load(file)

path_original="/workspace/data/full_metadata.csv"
meta = pd.read_csv(path_original, header=[0]).drop(columns=["Unnamed: 0", "Unnamed: 0.1", "Usage_original"])

In [3]:
config["data"]

{'data_dir': './data/',
 'data_format': 'csv',
 'metadata_path': './data/full_metadata.csv',
 'input_atlases': ['neuromorphometrics', 'suit', 'lpba40', 'cobra'],
 'diagnoses': ['HC, orange', 'MDD, blue', 'SCHZ, red', 'CTT, navy'],
 'resampling': 'hybrid',
 'resampling_method': '',
 'mtype': 'None',
 'train_proportion': 0.7,
 'train_test_split': {'remove_within_variable': ['Dataset'],
  'remove_variable_values': [['whiteCAT', 'NSS']],
  'test_on_subset': 'all',
  'extra_metadata_paths': ['/workspace/data/NSS_metadata.csv',
   '/workspace/data/whiteCAT_metadata.csv'],
  'train_valid_proportion': 0.8}}

In [10]:
plot_colors = {}
for i in range(0, len(config["data"]["diagnoses"]), 2):
    plot_colors[config["data"]["diagnoses"][i]] = config["data"]["diagnoses"][i+1]
plot_colors

{'HC, orange': 'MDD, blue', 'SCHZ, red': 'CTT, navy'}

In [4]:
split = config["data"]["train_test_split"]
split

{'remove_within_variable': ['Dataset', 'Sex'],
 'remove_variable_values': [['whiteCAT', 'NSS'], ['Male']],
 'test_on_subset': 'NSS',
 'extra_metadata_paths': ['/workspace/data/NSS_metadata.csv',
  '/workspace/data/whiteCAT_metadata.csv'],
 'train_valid_proportion': 0.8,
 'train_proportion': 0.7}

In [ ]:
def determine_split(split_config: dict, meta: pd.DataFrame):

    meta_split = {}

    if len(split_config["remove_within_variable"]) == 0:

        train_valid_df, test_df = split_annotations(annotations=meta, proportion=split_config["train_valid_proportion"])
        meta_split =  {"train_valid": train_valid_df, "test": test_df,}

    else: 
        removed_subsets = []
        
        for idx, var in enumerate(split_config["remove_within_variable"]): 
            remove_subset = meta[meta[var].isin(split_config["remove_variable_values"][idx])]
            removed_subsets.append(remove_subset)

        removed_data = pd.concat(removed_subsets).drop_duplicates(keep="first")
        removed_samples = removed_data["Filename"].to_list()

        filtered_df = meta[~meta["Filename"].isin(removed_samples)]
        
        train_valid_df, test_df = split_annotations(annotations=filtered_df, proportion=split_config["train_valid_proportion"])
        meta_split =  {"train_valid": train_valid_df, "test": test_df,}

        if split_config["test_on_subset"] == "all":
            meta_split["hold_out_test"] = removed_data
        else: 
            if split_config["test_on_subset"] in split_config["remove_within_variable"]:
                sub_idx = split_config["remove_within_variable"].index(split_config["test_on_subset"])
                meta_split["hold_out_test"] = removed_subsets[sub_idx]
            else: 
                for sub_idx, var_vals in enumerate(split_config["remove_variable_values"]): 
                    if split_config["test_on_subset"] in var_vals:
                        _hold_out = removed_subsets[sub_idx]

                        hold_out = _hold_out[_hold_out[split_config["remove_within_variable"][sub_idx]].isin([split_config["test_on_subset"]])]
                       
                        meta_split["hold_out_test"] = hold_out

    for purpose, df in meta_split.items(): 
        print(f"\n Shuffled metadata for {purpose}")
        print(f"Shape: {df.shape}")
        print(f"{df.head(5)}")

    return meta_split


In [ ]:
out = determine_split(split, meta)


 Shuffled metadata for train_valid
Shape: (908, 6)
              Filename Dataset Diagnosis   Age     Sex  Sex_int
0  IXI078-Guys-0751-T1     IXI        HC  47.4  Female        0
1  IXI140-Guys-0787-T1     IXI        HC  46.7  Female        0
2  IXI321-Guys-0903-T1     IXI        HC  73.1  Female        0
3  IXI312-Guys-0887-T1     IXI        HC  77.5  Female        0
4  IXI193-Guys-0810-T1     IXI        HC  74.0  Female        0

 Shuffled metadata for test
Shape: (226, 6)
              Filename Dataset Diagnosis   Age     Sex  Sex_int
0  IXI397-Guys-0953-T1     IXI        HC  69.8  Female        0
1    IXI226-HH-1618-T1     IXI        HC  60.6  Female        0
2  IXI403-Guys-0965-T1     IXI        HC  71.6  Female        0
3   IXI290-IOP-0874-T1     IXI        HC  64.2  Female        0
4    IXI102-HH-1416-T1     IXI        HC  43.5  Female        0

 Shuffled metadata for hold_out_test
Shape: (117, 6)
                   Filename Dataset Diagnosis   Age     Sex  Sex_int
2748  sub-NS

In [4]:
atlases = config["data"]["input_atlases"]
atlases

['neuromorphometrics', 'suit', 'lpba40', 'cobra']

In [ ]:
def get_atlas_map(atlases):
    atlas_map = {}
    for atlas in sorted(atlases): 
        caps = atlas.title()

        if f"[{caps[0]}]" in atlas_map.keys():
            atlas_map[f"[{caps[:1]}]"] = atlas
        else: 
            atlas_map[f"[{caps[0]}]"] = atlas
    return atlas_map

